# QuantumTX AH — Deep EDA Notebook

Comprehensive exploratory analysis of the 2024 Alexandra Hospital dataset.
Covers responder profiling, dropout analysis, dosage-response, comorbidity
patterns, and feature correlations.

**Run top-to-bottom.** All artefacts (PNGs, CSVs) are saved to `reports/eda_artefacts/`.

---

## Section 0 — Setup & Data Load

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend — required for save_fig
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import Markdown, display

# Project root — works whether kernel CWD is project root or notebooks/
PROJECT_ROOT = Path(".").resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

ARTEFACTS = PROJECT_ROOT / "reports" / "eda_artefacts"
ARTEFACTS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
PALETTE = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2","#937860","#DA8BC3","#8C8C8C"]

def save_fig(name: str):
    """Save current matplotlib figure to artefacts dir and display inline."""
    plt.savefig(ARTEFACTS / name, dpi=150, bbox_inches="tight")
    print(f"Saved: {ARTEFACTS / name}")
    plt.show()
    plt.close("all")

print("Setup complete.")


Setup complete.


In [2]:
df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "featured.parquet")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Derived subsets used throughout
df_followup      = df[df["has_followup"] == "Y"].copy()
df_dropout_known = df[df["is_dropout"].notna()].copy()

# Column groups
HAS_FLAGS   = sorted([c for c in df.columns if c.startswith("has_") and c != "has_followup"])
GRP_FLAGS   = sorted([c for c in df.columns if c.startswith("grp_")])
RGN_FLAGS   = sorted([c for c in df.columns if c.startswith("rgn_")])
IMPROV_COLS = [c for c in ["vas_improvement","tug_improvement","sst_improvement",
                             "normal_gs_improvement","fast_gs_improvement","sppb_improvement"]
               if c in df.columns]
_ALL_IMPROV_LABELS = {
    "vas_improvement":       "VAS Pain",
    "tug_improvement":       "TUG",
    "sst_improvement":       "5xSST",
    "normal_gs_improvement": "Normal GS",
    "fast_gs_improvement":   "Fast GS",
    "sppb_improvement":      "SPPB",
}
IMPROV_LABELS = {k: v for k, v in _ALL_IMPROV_LABELS.items() if k in IMPROV_COLS}
# MCID thresholds. None = percentage-based (computed per-row in analysis cells).
MCID_THRESHOLDS = {
    "vas_improvement":       2.0,    # ≥2 point reduction in VAS (0-10 scale)
    "tug_improvement":       3.0,    # ≥3 s improvement in TUG
    "sst_improvement":       None,   # ≥10% of pre_5xsst_s (relative — see Section 4)
    "normal_gs_improvement": 0.05,   # ≥0.05 m/s improvement
    "fast_gs_improvement":   0.10,   # ≥0.10 m/s improvement
    "sppb_improvement":      1.0,    # ≥1 point improvement in SPPB
}
NUMERIC_FEATS = [c for c in ["age","baseline_sppb","pre_normal_gs_ms","pre_tug_s",
                               "pre_5xsst_s","pre_vas","pre_fast_gs_ms",
                               "n_flags","n_regions","n_groups"] if c in df.columns]

# Reproduce README overview stats
total_n      = len(df)
n_followup   = (df["has_followup"] == "Y").sum()
n_responders = int((df_followup["overall_responder"] == 1).sum())
print(f"\nOverview:")
print(f"  Total patients : {total_n:,}")
print(f"  With follow-up : {n_followup:,}  ({n_followup/total_n*100:.1f}%)")
print(f"  Responders     : {n_responders:,}  ({n_responders/n_followup*100:.1f}% of follow-up)")
print(f"  Dropouts       : {int(df['is_dropout'].sum()):,}")


Loaded: 1,716 rows x 161 columns

Overview:
  Total patients : 1,716
  With follow-up : 596  (34.7%)
  Responders     : 414  (69.5% of follow-up)
  Dropouts       : 1,120


---
## Section 1 — Cohort Profiles

How are patients distributed across cohorts? We profile each cohort by size,
age, gender, follow-up rate, and responder rate to establish baseline context
for all later analyses.

In [3]:
# Build usage frequency short labels for column names
USAGE_MAP = {
    "Once (1x/week, one leg)":               "pct_1x",
    "Twice (2x/week, one leg per session)":  "pct_2x",
    "L+R 10 (20-min session, 10 min each leg)": "pct_lr",
}

rows = []
for cohort in sorted(df["cohort"].dropna().unique()):
    sub    = df[df["cohort"] == cohort]
    sub_fu = sub[sub["has_followup"] == "Y"]
    n_sub  = len(sub)
    row = {
        "cohort":            cohort,
        "n":                 n_sub,
        "pct_of_total":      round(n_sub / len(df) * 100, 1) if len(df) else float("nan"),
        "mean_age":          round(sub["age"].mean(), 1),
        "std_age":           round(sub["age"].std(), 1),
        "pct_female":        round((sub["gender"] == "F").sum() / n_sub * 100, 1) if n_sub else float("nan"),
        "n_followup":        len(sub_fu),
        "pct_followup":      round(len(sub_fu) / n_sub * 100, 1) if n_sub else float("nan"),
        "pct_responder":     round((sub_fu["overall_responder"] == 1).sum() / len(sub_fu) * 100, 1)
                             if len(sub_fu) >= 5 else float("nan"),  # suppress rate for very small sub-cohorts
    }
    # Usage frequency breakdown per cohort
    for freq_val, col_name in USAGE_MAP.items():
        row[col_name] = round((sub["usage_frequency"] == freq_val).sum() / n_sub * 100, 1) if n_sub else float("nan")
    rows.append(row)

cohort_df = pd.DataFrame(rows)
cohort_df.to_csv(ARTEFACTS / "cohort_profiles.csv", index=False)
display(cohort_df)


,cohort,n,pct_of_total,mean_age,std_age,pct_female,n_followup,pct_followup,pct_responder,pct_1x,pct_2x,pct_lr
0,Frailty/Sarcopenia,80,4.7,75.0,10.1,38.8,61,76.2,67.2,56.2,2.5,3.8
1,Neurological,76,4.4,70.7,11.4,47.4,63,82.9,74.6,75.0,2.6,0.0
2,Other-Mixed,17,1.0,73.4,11.0,52.9,12,70.6,83.3,64.7,0.0,0.0
3,Pain & Musculoskeletal,456,26.6,67.6,12.2,60.1,328,71.9,68.3,56.1,4.6,2.9
4,Post-Surgical/Rehab,52,3.0,70.2,12.9,55.8,39,75.0,82.1,63.5,7.7,1.9
5,Unclassified,1032,60.1,69.1,12.6,10.6,91,8.8,64.8,1.9,0.3,6.9
6,Wellness,3,0.2,70.5,6.4,33.3,2,66.7,NaN,66.7,0.0,0.0


In [4]:
cohort_colors = [PALETTE[i % len(PALETTE)] for i in range(len(cohort_df))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(cohort_df["cohort"], cohort_df["n"], color=cohort_colors)
axes[0].set_title("Patients per Cohort")
axes[0].set_ylabel("N")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(cohort_df["cohort"], cohort_df["pct_followup"], color=cohort_colors)
axes[1].axhline(n_followup / total_n * 100, color="red", linestyle="--", label="Overall avg")
axes[1].set_title("Follow-up Rate by Cohort")
axes[1].set_ylabel("% with Follow-up")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend()

plt.tight_layout()
save_fig("cohort_overview.png")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/cohort_overview.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Note:** ~60% of patients are classified as Unclassified (no comorbidity tags assigned). Subgroup analyses within named cohorts (Frailty, Neurological, etc.) are based on smaller samples and should be interpreted with caution.

---
## Section 2 — Responder Rates by Subgroup

Which patient groups respond best? We compute the percentage of overall_responder
(among follow-up patients) for every meaningful subgroup: cohort, age band, gender,
usage frequency, and each comorbidity flag.

In [5]:
def responder_rate(subdf):
    """% overall_responder among follow-up patients in subdf. Returns (rate, n_followup)."""
    fu = subdf[subdf["has_followup"] == "Y"]
    if len(fu) < 5:
        return float("nan"), len(fu)
    return round((fu["overall_responder"] == 1).sum() / len(fu) * 100, 1), len(fu)

rows = []
for val in sorted(df["cohort"].dropna().unique()):
    r, n = responder_rate(df[df["cohort"] == val])
    rows.append({"group": "cohort", "value": val, "n_followup": n, "responder_rate_pct": r})

for val in sorted(df["age_band"].dropna().unique()):
    r, n = responder_rate(df[df["age_band"] == val])
    rows.append({"group": "age_band", "value": str(val), "n_followup": n, "responder_rate_pct": r})

for val in [v for v in df["gender"].dropna().unique() if v != "__missing__"]:
    r, n = responder_rate(df[df["gender"] == val])
    rows.append({"group": "gender", "value": val, "n_followup": n, "responder_rate_pct": r})

for val in sorted(df["usage_frequency"].dropna().unique()):
    if val == "__missing__":
        continue
    r, n = responder_rate(df[df["usage_frequency"] == val])
    rows.append({"group": "usage_frequency", "value": str(val), "n_followup": n, "responder_rate_pct": r})

for flag in HAS_FLAGS:
    r, n = responder_rate(df[df[flag] == 1])
    rows.append({"group": "comorbidity_flag", "value": flag, "n_followup": n, "responder_rate_pct": r})

subgroup_df = pd.DataFrame(rows)
suppressed = subgroup_df[subgroup_df["responder_rate_pct"].isna()]
if len(suppressed):
    print(f"Suppressed {len(suppressed)} subgroups with n_followup < 5: {suppressed['value'].tolist()}")
subgroup_df = subgroup_df.dropna(subset=["responder_rate_pct"])
subgroup_df.to_csv(ARTEFACTS / "responder_rates_by_subgroup.csv", index=False)
display(subgroup_df.sort_values("responder_rate_pct", ascending=False).head(20))


Suppressed 9 subgroups with n_followup < 5: ['Wellness', 'has_cancer', 'has_cardiovascular', 'has_chronic_pain', 'has_fall_risk', 'has_fracture', 'has_frailty', 'has_neuropathy', 'has_wellness_only']


,group,value,n_followup,responder_rate_pct
27,comorbidity_flag,has_hypertension,15,93.3
22,comorbidity_flag,has_diabetes,26,84.6
29,comorbidity_flag,has_metabolic,26,84.6
35,comorbidity_flag,has_post_surgery,38,84.2
2,cohort,Other-Mixed,12,83.3
33,comorbidity_flag,has_osteoporosis,17,82.4
4,cohort,Post-Surgical/Rehab,39,82.1
18,comorbidity_flag,has_balance_issue,39,82.1
32,comorbidity_flag,has_oa,68,77.9
16,usage_frequency,"Twice (2x/week, one leg per session)",31,77.4


In [6]:
flag_df = (subgroup_df[subgroup_df["group"] == "comorbidity_flag"]
           .sort_values("responder_rate_pct", ascending=True))
overall_avg = (df_followup["overall_responder"] == 1).sum() / len(df_followup) * 100

fig, ax = plt.subplots(figsize=(8, max(5, len(flag_df) * 0.38)))
colors = ["#C44E52" if r < 50 else "#4C72B0" if r > 80 else "#DD8452"
          for r in flag_df["responder_rate_pct"]]
ax.barh(flag_df["value"].str.replace("has_", ""), flag_df["responder_rate_pct"], color=colors)
ax.axvline(overall_avg, color="black", linestyle="--", label=f"Overall avg ({overall_avg:.0f}%)")
ax.axvline(50, color="#C44E52", linestyle=":", alpha=0.6, label="50% threshold (low)")
ax.axvline(80, color="#4C72B0", linestyle=":", alpha=0.6, label="80% threshold (high)")
ax.set_xlabel("Responder Rate (%)")
ax.set_title("Responder Rate by Comorbidity Flag\n(red = <50%, orange = mid, blue = >80%)")
ax.legend(fontsize=8)
plt.tight_layout()
save_fig("responder_rates_by_flag.png")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/responder_rates_by_flag.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 3 — Responder vs Non-Responder Profiles

We compare baseline characteristics between responders and non-responders
using Mann-Whitney U tests (non-parametric, appropriate for skewed clinical data).

In [7]:
resp     = df_followup[df_followup["overall_responder"] == 1]
non_resp = df_followup[df_followup["overall_responder"] == 0]
print(f"Responders: {len(resp)}, Non-responders: {len(non_resp)}")

# Spec-defined feature list (Section 3 analysis only)
S3_FEATS = [c for c in ["age", "baseline_sppb", "pre_normal_gs_ms", "pre_tug_s",
                          "pre_5xsst_s", "pre_vas", "pre_fast_gs_ms", "n_flags"]
             if c in df_followup.columns]

rows = []
for feat in S3_FEATS:
    a = resp[feat].dropna()
    b = non_resp[feat].dropna()
    if len(a) < 5 or len(b) < 5:
        continue
    _, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append({
        "feature":            feat,
        "responder_mean":     round(a.mean(), 2),
        "responder_std":      round(a.std(), 2),
        "non_responder_mean": round(b.mean(), 2),
        "non_responder_std":  round(b.std(), 2),
        "p_value":            round(p, 4),
        "significant_p05":    p < 0.05,
    })

comp_df = pd.DataFrame(rows).sort_values("p_value")
comp_df.to_csv(ARTEFACTS / "responder_profile_comparison.csv", index=False)
display(comp_df)


Responders: 414, Non-responders: 182


,feature,responder_mean,responder_std,non_responder_mean,non_responder_std,p_value,significant_p05
1,baseline_sppb,9.44,2.52,9.98,3.11,0.0000,True
4,pre_5xsst_s,14.92,7.42,12.20,14.31,0.0000,True
6,pre_fast_gs_ms,1.20,0.44,1.38,0.44,0.0000,True
3,pre_tug_s,13.05,9.12,16.55,42.78,0.0001,True
2,pre_normal_gs_ms,0.88,0.30,0.94,0.36,0.0073,True
0,age,71.66,10.26,68.58,12.41,0.0088,True
7,n_flags,1.20,1.13,0.97,0.92,0.0460,True
5,pre_vas,5.00,1.76,4.76,1.82,0.2838,False


In [8]:
top_feats = comp_df.head(4)["feature"].tolist()
n_top = len(top_feats)
if n_top == 0:
    print("No features with sufficient data for violin plots.")
else:
    fig, axes = plt.subplots(1, n_top, figsize=(n_top * 4, 5))
    if n_top == 1:
        axes = [axes]
    for ax, feat in zip(axes, top_feats):
        data = df_followup[[feat, "overall_responder"]].dropna().copy()
        data["Responder"] = data["overall_responder"].map({1: "Yes", 0: "No"})
        sns.violinplot(data=data, x="Responder", y=feat, ax=ax,
                       palette={"Yes": "#4C72B0", "No": "#DD8452"}, inner="box")
        p_val = comp_df.loc[comp_df["feature"] == feat, "p_value"].values
        title = feat.replace("_", " ").title()
        if len(p_val):
            title += f"\np={p_val[0]:.4f}"
        ax.set_title(title)
        ax.set_xlabel("")
    plt.suptitle("Top Differentiating Features: Responders vs Non-Responders", fontsize=12, y=1.02)
    plt.tight_layout()
    save_fig("responder_profiles.png")


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/3398899085.py:12: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=data, x="Responder", y=feat, ax=ax,


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/responder_profiles.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/3398899085.py:12: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=data, x="Responder", y=feat, ax=ax,
/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/3398899085.py:12: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=data, x="Responder", y=feat, ax=ax,
/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/3398899085.py:12: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=data, x="Responder", y=feat, ax=ax,
/var/folders/fp/c3z_xxp9245c

---
## Section 4 — Individual Test Responder Rates

`overall_responder` requires hitting MCID on ≥ 2 tests. Here we look at each test
independently — which tests show the most improvement, and does this vary by cohort?
MCID thresholds: VAS ≥2, TUG ≥3s, 5xSST ≥10% of pre-score (relative), Normal GS ≥0.05 m/s,
Fast GS ≥0.10 m/s, SPPB ≥1.

In [9]:
rows = []
for col in IMPROV_COLS:
    threshold = MCID_THRESHOLDS.get(col)
    if threshold is None:
        # SST uses a relative threshold: ≥10% of pre-score
        sub_valid = df_followup[df_followup[col].notna() & df_followup["pre_5xsst_s"].notna()]
        hits = (sub_valid[col] >= sub_valid["pre_5xsst_s"] * 0.10).sum()
        sub  = sub_valid
    else:
        sub  = df_followup[df_followup[col].notna()]
        hits = (sub[col] >= threshold).sum()
    rows.append({
        "test":              IMPROV_LABELS.get(col, col),
        "n_paired":          len(sub),
        "n_mcid_hit":        int(hits),
        "mcid_hit_rate_pct": round(hits / len(sub) * 100, 1) if len(sub) > 0 else float("nan"),
        "mean_improvement":  round(sub[col].mean(), 3),
        "std_improvement":   round(sub[col].std(), 3),
    })

test_df = pd.DataFrame(rows).sort_values("mcid_hit_rate_pct", ascending=False)
test_df.to_csv(ARTEFACTS / "per_test_responder_rates.csv", index=False)
display(test_df)


,test,n_paired,n_mcid_hit,mcid_hit_rate_pct,mean_improvement,std_improvement
3,Normal GS,552,321,58.2,0.098,0.174
2,5xSST,515,294,57.1,2.017,5.192
0,VAS Pain,217,102,47.0,1.808,2.093
4,Fast GS,532,244,45.9,0.090,0.214
5,SPPB,561,248,44.2,0.627,1.186
1,TUG,546,111,20.3,1.990,6.601


In [10]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = [PALETTE[i % len(PALETTE)] for i in range(len(test_df))]
bars = ax.bar(test_df["test"], test_df["mcid_hit_rate_pct"], color=colors)
ax.axhline(
    test_df["mcid_hit_rate_pct"].mean(), color="black", linestyle="--",
    label=f"Mean ({test_df['mcid_hit_rate_pct'].mean():.0f}%)"
)
for bar, n in zip(bars, test_df["n_paired"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"n={n}", ha="center", va="bottom", fontsize=8)
ax.set_title("Per-Test MCID Hit Rate (% of patients with paired data)")
ax.set_ylabel("MCID Hit Rate (%)")
ax.set_xlabel("Test")
ax.tick_params(axis="x", rotation=20)
ax.legend()
plt.tight_layout()
save_fig("per_test_bar.png")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/per_test_bar.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
import numpy as np

cohorts  = sorted(df_followup["cohort"].dropna().unique())
hm_vals  = []
hm_annot = []
for col in IMPROV_COLS:
    threshold = MCID_THRESHOLDS.get(col)
    row, annot_row = [], []
    for cohort in cohorts:
        sub = df_followup[(df_followup["cohort"] == cohort) & df_followup[col].notna()]
        if threshold is None:
            # SST relative threshold
            sub = sub[sub["pre_5xsst_s"].notna()]
            rate_val = (sub[col] >= sub["pre_5xsst_s"] * 0.10).sum() / len(sub) * 100 if len(sub) >= 5 else float("nan")
        else:
            rate_val = (sub[col] >= threshold).sum() / len(sub) * 100 if len(sub) >= 5 else float("nan")
        if pd.isna(rate_val):
            row.append(float("nan"))
            annot_row.append(f"n={len(sub)}")
        else:
            row.append(round(rate_val, 1))
            annot_row.append(f"{rate_val:.0f}%\nn={len(sub)}")
    hm_vals.append(row)
    hm_annot.append(annot_row)

hm_df = pd.DataFrame(hm_vals,
                     index=[IMPROV_LABELS.get(c, c) for c in IMPROV_COLS],
                     columns=cohorts)

fig, ax = plt.subplots(figsize=(max(10, len(cohorts) * 1.6), 5))
sns.heatmap(hm_df, annot=pd.DataFrame(hm_annot, index=hm_df.index, columns=cohorts),
            fmt="", cmap="RdYlGn", ax=ax, vmin=0, vmax=100,
            mask=hm_df.isna(), linewidths=0.5,
            cbar_kws={"label": "MCID Hit Rate (%)"})
ax.set_title("Per-Test MCID Hit Rate (%) by Cohort  (grey = n < 5)")
ax.set_xlabel("Cohort")
ax.set_ylabel("Test")
plt.tight_layout()
save_fig("per_test_heatmap.png")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/per_test_heatmap.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 5 — Baseline Severity vs Improvement

Do patients with worse baseline scores improve more? A strong negative Spearman r
means sicker patients improve more — a regression-to-the-mean effect that weakens
predictive signal. We check this for each of the 6 tests.

In [12]:
import numpy as np

PAIRS = [
    ("pre_vas",          "vas_improvement",       "VAS Pain"),
    ("pre_tug_s",        "tug_improvement",       "TUG (s)"),
    ("pre_5xsst_s",      "sst_improvement",       "5xSST (s)"),
    ("pre_normal_gs_ms", "normal_gs_improvement", "Normal GS (m/s)"),
    ("pre_fast_gs_ms",   "fast_gs_improvement",   "Fast GS (m/s)"),
    ("baseline_sppb",    "sppb_improvement",      "SPPB"),
]
valid_pairs = [(pre, imp, lbl) for pre, imp, lbl in PAIRS
               if pre in df.columns and imp in df.columns]

cols_n = 3
rows_n = (len(valid_pairs) + cols_n - 1) // cols_n
fig, axes = plt.subplots(rows_n, cols_n, figsize=(cols_n * 5, rows_n * 4))
axes = axes.flatten() if rows_n > 1 else [axes] if cols_n == 1 else list(axes)

from matplotlib.lines import Line2D
for ax, (pre_col, imp_col, label) in zip(axes, valid_pairs):
    sub = df_followup[[pre_col, imp_col, "overall_responder"]].dropna()
    colors = sub["overall_responder"].map({1: "#4C72B0", 0: "#DD8452"})
    ax.scatter(sub[pre_col], sub[imp_col], c=colors, alpha=0.45, s=22)
    if len(sub) >= 3:
        slope, intercept, *_ = stats.linregress(sub[pre_col], sub[imp_col])
        x_range = np.linspace(sub[pre_col].min(), sub[pre_col].max(), 100)
        ax.plot(x_range, slope * x_range + intercept, color="black", linewidth=1.5)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    r, p = stats.spearmanr(sub[pre_col], sub[imp_col])
    ax.set_title(f"{label}\nSpearman r={r:.2f}, p={p:.3f}")
    ax.set_xlabel(f"Baseline ({pre_col})")
    ax.set_ylabel("Improvement")

for ax in axes[len(valid_pairs):]:
    ax.set_visible(False)

legend_handles = [
    Line2D([0],[0],marker="o",color="w",markerfacecolor="#4C72B0",markersize=8,label="Responder"),
    Line2D([0],[0],marker="o",color="w",markerfacecolor="#DD8452",markersize=8,label="Non-responder"),
]
fig.legend(handles=legend_handles, loc="lower right", fontsize=10)
plt.suptitle("Baseline Severity vs Improvement (negative r = regression to mean)", fontsize=13, y=1.01)
plt.tight_layout()
save_fig("baseline_vs_improvement.png")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/baseline_vs_improvement.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpreting the plots:**

- **Negative Spearman r (regression to mean):** If r < 0, patients with worse baseline scores improve more on average. This is clinically expected for pain (VAS) and gait speed — severely limited patients have more room to improve. A strong regression-to-mean effect (r < −0.3) means the baseline score is partly predicting improvement through ceiling/floor mechanics rather than true treatment response.

- **Positive or near-zero r:** If r ≥ 0, better baseline function does not predict less improvement — the treatment lifts all patients similarly regardless of starting point. This is a stronger signal of genuine treatment efficacy.

- **Ceiling effects:** Look for clusters of low-baseline patients (left side of scatter) with near-zero improvement — patients already near maximum impairment may have limited measurable change on that test's scale.

Check the Spearman r values above to identify which tests show the strongest regression-to-mean effect. Tests with r < −0.3 should be used cautiously as isolated predictors in the model — the improvement score conflates baseline severity with treatment response.

---
## Section 6 — Dropout Analysis

65% of patients have no follow-up. Is this random (MCAR) or patterned on baseline
characteristics? We compare dropouts vs completers and check dropout rates across
cohort, gender, and usage frequency.

In [13]:
dropouts  = df_dropout_known[df_dropout_known["is_dropout"] == 1]
completers = df_dropout_known[df_dropout_known["is_dropout"] == 0]
print(f"Dropouts: {len(dropouts)}, Completers: {len(completers)}")

# Same 8 spec features as Section 3
S6_FEATS = [c for c in ["age", "baseline_sppb", "pre_normal_gs_ms", "pre_tug_s",
                          "pre_5xsst_s", "pre_vas", "pre_fast_gs_ms", "n_flags"]
             if c in df_dropout_known.columns]

rows = []
for feat in S6_FEATS:
    a = dropouts[feat].dropna()
    b = completers[feat].dropna()
    if len(a) < 5 or len(b) < 5:
        continue
    _, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append({
        "feature":            feat,
        "dropout_mean":       round(a.mean(), 2),
        "dropout_std":        round(a.std(), 2),
        "completer_mean":     round(b.mean(), 2),
        "completer_std":      round(b.std(), 2),
        "p_value":            round(p, 4),
        "significant_p05":    p < 0.05,
    })

drop_comp_df = pd.DataFrame(rows).sort_values("p_value")
drop_comp_df.to_csv(ARTEFACTS / "dropout_profile_comparison.csv", index=False)
display(drop_comp_df)


Dropouts: 1120, Completers: 596


,feature,dropout_mean,dropout_std,completer_mean,completer_std,p_value,significant_p05
0,age,63.45,14.58,70.73,11.03,0.0000,True
1,baseline_sppb,10.43,2.67,9.60,2.72,0.0000,True
6,n_flags,0.18,0.55,1.13,1.08,0.0000,True
4,pre_5xsst_s,11.83,7.55,14.20,9.79,0.0003,True
2,pre_normal_gs_ms,1.01,0.35,0.90,0.32,0.0026,True
5,pre_fast_gs_ms,1.39,0.49,1.25,0.44,0.0047,True
3,pre_tug_s,11.63,11.93,14.01,23.73,0.0053,True


In [14]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
overall_dropout_rate = df_dropout_known["is_dropout"].mean() * 100

for ax, (col, title) in zip(axes, [
    ("cohort",           "Dropout Rate by Cohort"),
    ("age_band",         "Dropout Rate by Age Band"),
    ("gender",           "Dropout Rate by Gender"),
    ("usage_frequency",  "Dropout Rate by Usage Frequency"),
]):
    vals = sorted([v for v in df_dropout_known[col].dropna().unique() if v != "__missing__"])
    rates = [(str(v), df_dropout_known[df_dropout_known[col]==v]["is_dropout"].mean()*100)
             for v in vals]
    rates.sort(key=lambda x: x[1], reverse=True)
    bar_colors = [PALETTE[i % len(PALETTE)] for i in range(len(rates))]
    ax.bar([r[0] for r in rates], [r[1] for r in rates], color=bar_colors)
    ax.axhline(overall_dropout_rate, color="red", linestyle="--", label=f"Avg {overall_dropout_rate:.0f}%")
    ax.set_title(title)
    ax.set_ylabel("Dropout Rate (%)")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=8)

plt.tight_layout()
save_fig("dropout_analysis.png")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/dropout_analysis.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Missingness by dropout status — does missing data predict dropout?
miss_rows = []
for feat in S6_FEATS:
    do_miss = dropouts[feat].isna().mean() * 100
    co_miss = completers[feat].isna().mean() * 100
    miss_rows.append({"feature": feat,
                      "dropout_missing_pct": round(do_miss, 1),
                      "completer_missing_pct": round(co_miss, 1),
                      "difference_pct": round(do_miss - co_miss, 1)})
miss_df = pd.DataFrame(miss_rows).sort_values("difference_pct", ascending=False)
display(miss_df)
print("\nPositive difference = dropouts have MORE missingness (potential MNAR — preview of Section 12)")


,feature,dropout_missing_pct,completer_missing_pct,difference_pct
2,pre_normal_gs_ms,93.5,4.5,89.0
3,pre_tug_s,93.7,5.4,88.3
1,baseline_sppb,87.9,3.2,84.7
6,pre_fast_gs_ms,91.8,7.9,83.9
0,age,85.8,2.5,83.3
4,pre_5xsst_s,94.9,12.4,82.5
5,pre_vas,100.0,61.2,38.8
7,n_flags,0.0,0.0,0.0



Positive difference = dropouts have MORE missingness (potential MNAR — preview of Section 12)


---
## Section 7 — Dosage-Response

Does treatment frequency predict outcomes? We compare composite improvement and
responder rate across the three usage groups. A Kruskal-Wallis test checks whether
the differences are statistically meaningful.

In [16]:
USAGE_VALID = [v for v in sorted(df_followup["usage_frequency"].dropna().unique())
               if v != "__missing__"]

rows = []
for usage in USAGE_VALID:
    sub  = df_followup[df_followup["usage_frequency"] == usage]
    comp = sub["composite_improvement"].dropna()
    rows.append({
        "usage_frequency":    str(usage)[:30],
        "n":                  len(sub),
        "n_with_composite":   len(comp),
        "mean_composite":     round(comp.mean(), 3) if len(comp) > 0 else float("nan"),
        "std_composite":      round(comp.std(), 3)  if len(comp) > 0 else float("nan"),
        "responder_rate_pct": round((sub["overall_responder"] == 1).sum() / len(sub) * 100, 1)
                              if len(sub) > 0 else float("nan"),
    })
dosage_df = pd.DataFrame(rows)
dosage_df.to_csv(ARTEFACTS / "dosage_response.csv", index=False)
display(dosage_df)

groups = [df_followup[df_followup["usage_frequency"]==u]["composite_improvement"].dropna().values
          for u in USAGE_VALID
          if len(df_followup[df_followup["usage_frequency"]==u]["composite_improvement"].dropna()) >= 5]
if len(groups) >= 2:
    h, p = stats.kruskal(*groups)
    print(f"\nKruskal-Wallis H={h:.3f}, p={p:.4f}")
    print("Significant" if p < 0.05 else "No significant difference between dosage groups (p >= 0.05)")
else:
    print("\nInsufficient groups for Kruskal-Wallis test.")

# Flag underpowered groups
underpowered = dosage_df[dosage_df["n"] < 20]
if len(underpowered):
    print(f"\nNote: {len(underpowered)} dosage group(s) have n < 20 — comparisons may be underpowered:")
    for _, row_u in underpowered.iterrows():
        print(f"  {row_u['usage_frequency']}: n={row_u['n']}, n_with_composite={row_u['n_with_composite']}")
else:
    print(f"\nSample sizes: {dict(zip(dosage_df['usage_frequency'], dosage_df['n']))}")
    print("Note: unequal group sizes may affect Kruskal-Wallis power — interpret cautiously.")


,usage_frequency,n,n_with_composite,mean_composite,std_composite,responder_rate_pct
0,"L+R 10 (20-min session, 10 min",58,58,0.054,0.655,69.0
1,"Once (1x/week, one leg)",423,423,0.004,0.690,70.2
2,"Twice (2x/week, one leg per se",31,31,-0.021,0.481,77.4



Kruskal-Wallis H=0.612, p=0.7364
No significant difference between dosage groups (p >= 0.05)

Sample sizes: {'L+R 10 (20-min session, 10 min': 58, 'Once (1x/week, one leg)': 423, 'Twice (2x/week, one leg per se': 31}
Note: unequal group sizes may affect Kruskal-Wallis power — interpret cautiously.


In [17]:
short_labels = [str(u).split("(")[0].strip()[:20] for u in USAGE_VALID]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

box_data = [df_followup[df_followup["usage_frequency"]==u]["composite_improvement"].dropna().values
            for u in USAGE_VALID]
bp = axes[0].boxplot(box_data, labels=short_labels, patch_artist=True)
for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_title("Composite Improvement by Dosage")
axes[0].set_ylabel("Composite Improvement (z-score)")
axes[0].tick_params(axis="x", rotation=20)

overall_rr = (df_followup["overall_responder"]==1).sum()/len(df_followup)*100
axes[1].bar(short_labels, dosage_df["responder_rate_pct"],
            color=[PALETTE[i % len(PALETTE)] for i in range(len(dosage_df))])
axes[1].axhline(overall_rr, color="red", linestyle="--", label=f"Overall avg ({overall_rr:.0f}%)")
axes[1].set_title("Responder Rate by Dosage")
axes[1].set_ylabel("Responder Rate (%)")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend()

plt.tight_layout()
save_fig("dosage_response.png")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/dosage_response.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/3457635829.py:6: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[0].boxplot(box_data, labels=short_labels, patch_artist=True)
/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Sample size caveat:** The three dosage groups (Once 1x/week, Twice 2x/week, L+R 10)
are likely unevenly distributed. Groups with n < 20 should be interpreted with caution —
the Kruskal-Wallis test above provides a formal test for group differences, but low-powered
groups may miss true effects or produce false signals. The p-value above determines
whether any observed difference is statistically reliable.

---
## Section 8 — Age × Condition Interactions

Does the same cohort respond differently across age groups? This heatmap reveals
interaction patterns a model with only main effects would miss. Cells with n < 10
are greyed out as underpowered.

In [18]:
import numpy as np

cohorts   = sorted(df_followup["cohort"].dropna().unique())
age_bands = sorted(df_followup["age_band"].dropna().unique())

hm_vals, hm_annot = [], []
for band in age_bands:
    row, annot_row = [], []
    for cohort in cohorts:
        sub = df_followup[(df_followup["age_band"]==band) & (df_followup["cohort"]==cohort)]
        if len(sub) < 10:
            row.append(float("nan"))
            annot_row.append(f"n={len(sub)}")
        else:
            rate = round((sub["overall_responder"]==1).sum()/len(sub)*100, 1)
            row.append(rate)
            annot_row.append(f"{rate:.0f}%\nn={len(sub)}")
    hm_vals.append(row)
    hm_annot.append(annot_row)

hm_df = pd.DataFrame(hm_vals, index=[str(b) for b in age_bands], columns=cohorts)
hm_df.to_csv(ARTEFACTS / "age_cohort_responder_rates.csv")

fig, ax = plt.subplots(figsize=(max(10, len(cohorts)*1.6), max(4, len(age_bands)*1.3)))
sns.heatmap(hm_df,
            annot=pd.DataFrame(hm_annot, index=hm_df.index, columns=cohorts),
            fmt="", cmap="RdYlGn", ax=ax, vmin=50, vmax=90,
            mask=hm_df.isna(), linewidths=0.5,
            cbar_kws={"label": "Responder Rate (%)"})
ax.set_title("Responder Rate (%) — Age Band × Cohort  (grey = n < 10, underpowered)")
ax.set_xlabel("Cohort")
ax.set_ylabel("Age Band")
plt.tight_layout()
save_fig("age_cohort_heatmap.png")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/age_cohort_heatmap.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 9 — Comorbidity Co-occurrence

Which conditions appear together? Jaccard similarity measures overlap:
|A ∩ B| / |A ∪ B|. High co-occurrence pairs may warrant interaction features.

In [19]:
import numpy as np

flag_mat = df[HAS_FLAGS].fillna(0).astype(int)
n_flags  = len(HAS_FLAGS)
jaccard  = np.zeros((n_flags, n_flags))
for i in range(n_flags):
    for j in range(n_flags):
        a = flag_mat.iloc[:, i].values.astype(bool)
        b = flag_mat.iloc[:, j].values.astype(bool)
        union = (a | b).sum()
        jaccard[i, j] = (a & b).sum() / union if union > 0 else 0.0

short_names = [f.replace("has_","") for f in HAS_FLAGS]
jac_df = pd.DataFrame(jaccard, index=short_names, columns=short_names)

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.eye(n_flags, dtype=bool)
sns.heatmap(jac_df, mask=mask, annot=False, cmap="Blues", ax=ax,
            vmin=0, vmax=0.4, linewidths=0.3,
            cbar_kws={"label": "Jaccard Similarity"})
ax.set_title("Comorbidity Co-occurrence (Jaccard Similarity)")
plt.tight_layout()
save_fig("comorbidity_cooccurrence.png")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/comorbidity_cooccurrence.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
pairs = []
for i in range(n_flags):
    for j in range(i+1, n_flags):
        if jaccard[i, j] > 0.01:
            co_n = int((flag_mat.iloc[:,i].astype(bool) & flag_mat.iloc[:,j].astype(bool)).sum())
            pairs.append({"flag_a": short_names[i], "flag_b": short_names[j],
                          "jaccard": round(jaccard[i,j], 3), "co_occurrence_n": co_n})
pairs_df = pd.DataFrame(pairs).sort_values("jaccard", ascending=False)
pairs_df.to_csv(ARTEFACTS / "comorbidity_cooccurrence_pairs.csv", index=False)
display(pairs_df.head(10))
print("\nHigh Jaccard pairs may benefit from interaction features in the model.")


,flag_a,flag_b,jaccard,co_occurrence_n
37,diabetes,metabolic,1.000,40
85,neurological,stroke,0.476,20
81,neurological,parkinsons,0.452,19
66,knee_issue,oa,0.125,43
35,diabetes,hypertension,0.100,5
60,hypertension,metabolic,0.100,5
100,post_surgery,spinal_issue,0.100,9
68,knee_issue,post_surgery,0.097,31
55,hip_issue,post_surgery,0.086,8
38,diabetes,neurological,0.079,6



High Jaccard pairs may benefit from interaction features in the model.


**Clinically expected high co-occurrence pairs** include conditions in the same clinical domain,
such as Osteoarthritis with other musculoskeletal conditions, or Parkinson's disease with neurological
fall-risk. **Unexpected pairs** — e.g. OA appearing with diabetes or respiratory conditions — may
reflect the broad frailty/sarcopenia phenotype where multi-system dysfunction clusters together.
High co-occurrence pairs (Jaccard > 0.1) are candidates for interaction features in the model.

---
## Section 10 — Comorbidity Burden vs Outcomes

Does having more conditions (higher n_flags) predict worse response?
A non-linear relationship here suggests binning n_flags may be useful as a feature.

In [21]:
import numpy as np

sub = df_followup[df_followup["composite_improvement"].notna() &
                  df_followup["n_flags"].notna()].copy()

# Cohort colour mapping for scatter
cohort_list = sorted(sub["cohort"].dropna().unique())
cohort_color_map = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(cohort_list)}

sub["flag_bucket"] = sub["n_flags"].apply(lambda x: "4+" if x >= 4 else str(int(x)))
bucket_order = [b for b in ["0","1","2","3","4+"] if b in sub["flag_bucket"].values]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 0: Scatter n_flags vs composite_improvement, coloured by cohort
for cohort in cohort_list:
    cohort_sub = sub[sub["cohort"] == cohort]
    axes[0].scatter(
        cohort_sub["n_flags"] + np.random.default_rng(42).uniform(-0.15, 0.15, len(cohort_sub)),
        cohort_sub["composite_improvement"],
        alpha=0.4, s=18, color=cohort_color_map[cohort], label=cohort
    )
r, p = stats.spearmanr(sub["n_flags"], sub["composite_improvement"])
if len(sub) >= 3:
    slope, intercept, *_ = stats.linregress(sub["n_flags"], sub["composite_improvement"])
    x_range = np.linspace(sub["n_flags"].min(), sub["n_flags"].max(), 100)
    axes[0].plot(x_range, slope*x_range+intercept, color="black", linewidth=1.5)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_title(f"n_flags vs Composite Improvement\nSpearman r={r:.2f}, p={p:.3f}")
axes[0].set_xlabel("n_flags")
axes[0].set_ylabel("Composite Improvement")
axes[0].legend(fontsize=6, loc="upper right", ncol=2)

# Panel 1: Box plot — composite improvement by n_flags bucket
box_data = [sub[sub["flag_bucket"]==b]["composite_improvement"].dropna().values
            for b in bucket_order]
bp = axes[1].boxplot(box_data, labels=bucket_order, patch_artist=True)
for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title("Composite Improvement by n_flags Bucket")
axes[1].set_xlabel("n_flags bucket")
axes[1].set_ylabel("Composite Improvement")

# Panel 2: Responder rate by n_flags bucket
rr_vals = []
for bucket in bucket_order:
    b_sub = sub[sub["flag_bucket"] == bucket]
    if len(b_sub) >= 5:
        rr_vals.append({"bucket": bucket, "n": len(b_sub),
                        "responder_rate": round((b_sub["overall_responder"]==1).sum()/len(b_sub)*100, 1)})
if rr_vals:
    rr_df = pd.DataFrame(rr_vals)
    axes[2].bar(rr_df["bucket"], rr_df["responder_rate"],
                color=[PALETTE[i % len(PALETTE)] for i in range(len(rr_df))])
    axes[2].set_title("Responder Rate by Condition Count")
    axes[2].set_xlabel("n_flags bucket")
    axes[2].set_ylabel("Responder Rate (%)")
else:
    axes[2].set_visible(False)

plt.tight_layout()
save_fig("comorbidity_burden.png")


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/2778914566.py:37: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[1].boxplot(box_data, labels=bucket_order, patch_artist=True)


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/comorbidity_burden.png


/var/folders/fp/c3z_xxp9245c7jwzpdr2n5f80000gn/T/ipykernel_42727/1004102212.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 11 — Feature Correlations

High correlations (|r| > 0.7) between features indicate potential redundancy and can
inflate SHAP values. We identify pairs worth reviewing before the next model iteration.

In [22]:
import numpy as np

all_numeric = df.select_dtypes(include="number").columns.tolist()
exclude = {"overall_responder", "composite_improvement", "is_dropout"}
exclude |= {c for c in all_numeric if c.endswith("_improvement")}
valid_numeric = [c for c in all_numeric if c not in exclude and df[c].nunique() > 2]

corr = df[valid_numeric].corr(method="spearman", numeric_only=True)

# Use clustermap for hierarchical clustering of features
g = sns.clustermap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                   figsize=(max(14, len(valid_numeric)*0.55), max(12, len(valid_numeric)*0.55)),
                   linewidths=0.1, annot=False,
                   cbar_kws={"label": "Spearman r", "shrink": 0.5},
                   dendrogram_ratio=0.1)
g.ax_heatmap.set_title("Spearman Correlation Matrix — Numeric Model Features (clustered)", pad=20)
g.savefig(ARTEFACTS / "feature_correlations.png", dpi=150, bbox_inches="tight")
print(f"Saved: {ARTEFACTS / 'feature_correlations.png'}")
plt.close("all")


Saved: /Users/reetmitra/Desktop/QTX/quantumtx-ah/reports/eda_artefacts/feature_correlations.png


In [23]:
high_corr = []
cols_list = corr.columns.tolist()
for i in range(len(cols_list)):
    for j in range(i+1, len(cols_list)):
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append({"feature_a": cols_list[i], "feature_b": cols_list[j],
                               "spearman_r": round(r, 3), "abs_r": round(abs(r), 3)})
high_corr_df = pd.DataFrame(high_corr).sort_values("abs_r", ascending=False) if high_corr else pd.DataFrame(
    columns=["feature_a","feature_b","spearman_r","abs_r"])
high_corr_df.to_csv(ARTEFACTS / "high_correlation_pairs.csv", index=False)

if high_corr_df.empty:
    print("No pairs with |Spearman r| > 0.7.")
else:
    print(f"{len(high_corr_df)} high-correlation pairs found (|r| > 0.7):")
    display(high_corr_df)


95 high-correlation pairs found (|r| > 0.7):


,feature_a,feature_b,spearman_r,abs_r
81,post_fast_time_s,post_fast_gs_ms,-1.000,1.000
76,pre_fast_time_s,pre_fast_gs_ms,-1.000,1.000
54,post_normal_time_s,post_normal_gs_ms,-1.000,1.000
45,pre_normal_time_s,pre_normal_gs_ms,-1.000,1.000
43,sst_change_pct,sst_change_computed_pct,-1.000,1.000
...,...,...,...,...
38,post_5xsst_s,post_fast_gs_ms,-0.725,0.725
36,post_5xsst_s,post_fast_time_s,0.724,0.724
35,post_5xsst_s,pre_fast_time_s,0.707,0.707
37,post_5xsst_s,pre_fast_gs_ms,-0.707,0.707


---
## Section 12 — Missingness Deep Dive

Is missingness associated with dropout (MNAR pattern)? If patients who are sicker
are both more likely to drop out AND more likely to have missing baseline data,
simple imputation strategies will produce biased features. We use chi-square tests.

In [24]:
df_miss = df[df["is_dropout"].notna()].copy()
feat_cols = [c for c in ["age","baseline_sppb","pre_normal_gs_ms","pre_tug_s",
                          "pre_5xsst_s","pre_vas","pre_fast_gs_ms"] if c in df.columns]

rows = []
for feat in feat_cols:
    miss_flag = df_miss[feat].isna().astype(int)
    dropout   = df_miss["is_dropout"].astype(int)
    ct = pd.crosstab(miss_flag, dropout)
    if ct.shape == (2, 2):
        chi2, p, _, _ = stats.chi2_contingency(ct)
    else:
        chi2, p = float("nan"), float("nan")
    pct_miss_do  = df_miss[df_miss["is_dropout"]==1][feat].isna().mean()*100
    pct_miss_co  = df_miss[df_miss["is_dropout"]==0][feat].isna().mean()*100
    rows.append({
        "feature":                feat,
        "pct_missing_dropout":    round(pct_miss_do, 1),
        "pct_missing_completer":  round(pct_miss_co, 1),
        "difference":             round(pct_miss_do - pct_miss_co, 1),
        "chi2":                   round(chi2, 3) if not pd.isna(chi2) else float("nan"),
        "p_value":                round(p, 4)    if not pd.isna(p)    else float("nan"),
        "mnar_signal":            bool(p < 0.05) if not pd.isna(p)    else False,
    })

miss_df = pd.DataFrame(rows).sort_values("difference", ascending=False)
miss_df.to_csv(ARTEFACTS / "missingness_by_dropout.csv", index=False)
display(miss_df)
mnar = miss_df[miss_df["mnar_signal"]==True]["feature"].tolist()
if mnar:
    print(f"\nMNAR signal detected for: {mnar}")
    print("These features have significantly higher missingness among dropouts.")
    print("Simple median imputation may be biased — consider missingness indicator features.")
else:
    print("\nNo MNAR signal detected at p < 0.05.")


,feature,pct_missing_dropout,pct_missing_completer,difference,chi2,p_value,mnar_signal
2,pre_normal_gs_ms,93.5,4.5,89.0,1310.681,0.0,True
3,pre_tug_s,93.7,5.4,88.3,1297.046,0.0,True
1,baseline_sppb,87.9,3.2,84.7,1144.782,0.0,True
6,pre_fast_gs_ms,91.8,7.9,83.9,1166.555,0.0,True
0,age,85.8,2.5,83.3,1096.758,0.0,True
4,pre_5xsst_s,94.9,12.4,82.5,1180.427,0.0,True
5,pre_vas,100.0,61.2,38.8,498.298,0.0,True



MNAR signal detected for: ['pre_normal_gs_ms', 'pre_tug_s', 'baseline_sppb', 'pre_fast_gs_ms', 'age', 'pre_5xsst_s', 'pre_vas']
These features have significantly higher missingness among dropouts.
Simple median imputation may be biased — consider missingness indicator features.


---
## Section 13 — Model-Readiness Summary

Consolidated findings from this EDA and their implications for model improvement.

### Feature Engineering Candidates

Review the section outputs above to identify specific actions:

- **Age × cohort interaction term** (Section 8): if the heatmap shows cells where a specific age band + cohort combination has notably different responder rates from the marginal rates, add `age_band_cohort` as an interaction feature.
- **Ordinal dosage encoding** (Section 7): if Kruskal-Wallis p < 0.05, replace one-hot usage_frequency with ordinal 1/2/3 (Once < Twice < L+R) as a numeric feature.
- **Baseline severity ratio** (Section 5): for tests with strong negative Spearman r (r < -0.3), consider adding `pre_score / cohort_mean_pre_score` to capture relative severity.
- **n_flags bucket** (Section 10): if responder rate shows a non-linear pattern, bin n_flags into 0 / 1 / 2 / 3+ buckets as a categorical feature alongside the raw count.
- **Comorbidity interaction pairs** (Section 9): the top 3 Jaccard pairs from `comorbidity_cooccurrence_pairs.csv` — consider adding their product (flag_a × flag_b) as a binary interaction feature.

### Features to Consider Removing

See `high_correlation_pairs.csv` (Section 11). Any pair with |r| > 0.85 where one feature
is a near-superset of the other (e.g., `grp_joint_disease` largely subsumes `has_oa`) —
consider dropping the redundant one to reduce noise in SHAP interpretation.

### Subgroups to Stratify On in CV

If Section 8 reveals strong cohort × age_band effects, add `cohort` to the stratification
key in `RandomizedSearchCV` (currently stratified on outcome label only).

### Known Confounders Not Currently Controlled

Sessions completed (adherence), BMI, and medications are the highest-priority missing
confounders. See Section 14 for the full data gap inventory.

---
## Section 14 — Data Gaps & Next Steps

The findings above are constrained by data availability. The questions below were
compiled during this analysis and are mapped to the sections they would improve.

In [25]:
from IPython.display import Markdown, display
questions_path = PROJECT_ROOT / "docs" / "data-questions.md"
if questions_path.exists():
    display(Markdown(questions_path.read_text()))
else:
    print(f"Not found: {questions_path}\nRun notebook from project root.")


# External Data Points to Request

Questions about data that isn't currently in the pipeline but would meaningfully improve
EDA depth and model performance. Grouped by category.

---

## 1. Treatment / Session Data

These are the highest-priority gaps. Session-level data would unlock time-series analysis,
adherence modelling, and dose-response curves.

| Question | Why it matters |
|---|---|
| How many sessions did each patient complete? | Strongest adherence signal; separates early dropouts from late dropouts |
| What were the session dates? | Enables treatment duration calculation and time-to-dropout analysis |
| What was the gap between baseline assessment and first session? | Long gaps may predict dropout |
| What was the gap between last session and follow-up assessment? | Affects post-measurement validity |
| Which leg(s) were treated in each session (1x protocol)? | Allows bilateral vs unilateral comparison |
| Were any sessions missed and then made up? | Distinguishes irregular attendance from pure dropout |
| Was there a standardised warm-up or cool-down protocol? | Controls for confounders in gait/functional tests |
| What device/machine ID was used? | Batch effects between machines at the centre |

---

## 2. Patient Characteristics

| Question | Why it matters |
|---|---|
| Height and weight (BMI)? | BMI is a known confounder for mobility and pain outcomes |
| Grip strength (hand dynamometry)? | Strong predictor of frailty and functional recovery |
| Cognitive status (MMSE or equivalent)? | Affects follow-up adherence and self-reported pain accuracy |
| Living situation (alone vs with family)? | Social support predicts adherence |
| Smoking status? | Affects muscle recovery and cardiovascular fitness |
| Employment / activity level? | Sedentary vs active baseline affects trajectory |
| Prior physiotherapy or similar treatment in the past 12 months? | Controls for concurrent care effects |
| Number of falls in the past 12 months? | Grounds the fall-risk flag in a quantitative measure |

---

## 3. Clinical / Medical Data

| Question | Why it matters |
|---|---|
| Comorbidity severity, not just presence (e.g., HbA1c for diabetes, Hoehn & Yahr for Parkinson's)? | Binary flags lose severity information that affects prognosis |
| Current medications (especially opioids, NSAIDs, beta-blockers, statins)? | Multiple drug classes confound pain, gait, and muscle response |
| Pain location specificity (which joint, bilateral vs unilateral)? | OA in knee vs hip vs shoulder have very different trajectories |
| Frailty scale score (e.g., FRAIL or CFS), not just binary flag? | Quantitative frailty gradient predicts recovery rate |
| Reason for referral (GP, specialist, self)? | Referral source correlates with severity and motivation |
| Any adverse events or treatment pauses reported? | Necessary for safety analysis and dropout attribution |
| Post-surgical type and time since surgery (for post-surgical cohort)? | Recency and procedure type drive expected recovery curve |

---

## 4. Dropout / Non-Completion

| Question | Why it matters |
|---|---|
| Was dropout voluntary (patient decision) or administrative (moved, cost, scheduling)? | Voluntary dropout may carry clinical signal; admin dropout is noise |
| Was a reason recorded for not completing follow-up assessment? | Separates "improved and discharged" from "lost to follow-up" |
| Were patients contacted after dropout? Any outcome data available? | Even partial post data would reduce outcome selection bias |
| Was there a planned endpoint different from the standard follow-up window? | Some patients may have had shorter programmes by design |

---

## 5. Programme / Operational Data

| Question | Why it matters |
|---|---|
| Referring department or physician? | Referral pathway may correlate with patient preparation and motivation |
| Subsidy or insurance type (MediShield Life, CHAS, self-pay)? | Cost burden predicts dropout in Singapore context |
| Time of day / day of week for sessions? | Scheduling patterns can proxy for lifestyle and motivation |
| Was this the patient's first QTX programme or a repeat? | Repeat patients have different baselines and expectations |
| Was a home exercise programme prescribed alongside QTX sessions? | Co-intervention must be controlled for in outcome modelling |

---

## 6. Mid-Programme Assessments

| Question | Why it matters |
|---|---|
| Are any mid-programme assessments available (e.g., at session 5 or 10)? | Enables early-response prediction and trajectory modelling |
| Are patient-reported outcomes (pain diaries, satisfaction scores) collected during the programme? | Richer signal than a single post-assessment |

---

## Priority Order

If only some of these can be obtained, prioritise in this order:

1. **Sessions completed + session dates** — unlocks adherence, dropout timing, dose-response
2. **Dropout reason** — resolves the biggest modelling ambiguity
3. **BMI** — quick to collect, high confounding impact
4. **Comorbidity severity scores** — upgrades binary flags to continuous features
5. **Medications** — major confounder currently uncontrolled


### Mapping: Data Gap → Section It Would Improve

| Data Point | Sections Improved |
|---|---|
| Sessions completed + session dates | §7 Dosage-Response (actual dose), §6 Dropout (timing of withdrawal) |
| Dropout reason (voluntary vs admin) | §6 Dropout Analysis (separates informative from noise dropout) |
| BMI | §3 Responder Profiles, §6 Dropout Analysis |
| Comorbidity severity scores | §9 Co-occurrence, §10 Burden vs Outcomes |
| Current medications | §3 Responder Profiles (as confounder) |
| Mid-programme assessments | §5 Baseline Severity (enables trajectory modelling) |
| Home exercise compliance | §7 Dosage-Response (true treatment dose) |

---
*End of Deep EDA Notebook — QuantumTX AH 2024*